In [ ]:
import os
import pickle
import csv
import re
import random
from collections import defaultdict
import torch
import pandas as pd
from tqdm import tqdm
from datasets import Dataset

# Transformers imports
from transformers import (
    GPT2Tokenizer,
    AutoModelForCausalLM,
    AutoTokenizer,
    GPTNeoForCausalLM,
    pipeline
)

# Scrubadub for PII detection
import scrubadub
import json

In [ ]:
gid = 0  
os.environ["CUDA_VISIBLE_DEVICES"] = f"{gid}"

if gid is not None:
    device = f"cuda:{gid}"
else:
    device = 'cpu'
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

decoding_alg = "greedy"  # "beam_search", "top_k", o "greedy"

torch.manual_seed(42)



contextToDo= ["context-200"]

pii_type = "phone" #"email_cc"#"twitter"
languages = ["eng"]

data_folder = "./data"
result_folder= f"./results"

os.makedirs(result_folder, exist_ok=True)

BATCH_SIZE = 32


redo = False

pd.set_option('display.max_colwidth', None)


In [ ]:
def load_json(filename):
        with open(filename, 'r', encoding='utf-8') as f:
            return json.load(f)

def save_json(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

In [ ]:
import json
import os
from pathlib import Path

env = json.loads(Path("env.json").read_text(encoding="utf-8"))
os.environ["HF_TOKEN"] = env["HF_TOKEN_ATTACKS"]

In [ ]:

def load_data(pii_type, filepath):
    """Carica e preprocessa un dataset PII."""
    data = Dataset.load_from_disk(filepath)
    data = pd.DataFrame(data)
    data['context'] = data['context'].apply(str.strip)
    # Sampling per dataset URL se troppo grande
    if len(data) > 4550 and pii_type == 'url':
        data = data.sample(n=4550, random_state=42).reset_index(drop=True)

    data = Dataset.from_pandas(data[['pii', 'context']])
    return data


def load_pickle(filename):
    
    with open(filename, "rb") as pickle_handler:
        results = pickle.load(pickle_handler)
    return results

In [ ]:
scrubber = scrubadub.Scrubber()
print("Detector disponibili:")
print(scrubber._detectors)

to_remove = []
for k in scrubber._detectors:
    if k not in pii_type:
        to_remove.append(scrubber._detectors[k])

for d in to_remove:
    scrubber.remove_detector(d)

print(f"\nDetector attivo per: {pii_type}")
print(scrubber._detectors)

In [ ]:
def pii_findall(predicted):
    
    all_preds = []
    for filth in scrubber.iter_filth(predicted):
        all_preds.append(filth.text)
    return all_preds


def get_prompts_context(dataset, k=100):
    
    contexts = {}
    for example in dataset:
        contexts[example['pii']] = example['context']

    prompts = []
    name_list = []

    for pii, context in tqdm(contexts.items()):
        tokens = tokenizer(context[-1000:])['input_ids']
        name_list.append(pii)
        prompt = tokenizer.decode(tokens[-k:])
        prompts.append(prompt)

    return prompts, name_list

In [ ]:
def load(pii_type, dataset_path):
    data = load_data(pii_type, dataset_path)
    return data

In [ ]:
from time import sleep

In [ ]:
model_types = {
    'Qwen2.5': ['3B', "7B"],
    'Llama-3.2': ['1B', '3B'],
    'gpt-neo': ['1.3B', '2.7B'],
    'gpt-j': ['6B']
}


for model_type in model_types:
    for model_size in model_types[model_type]:

        if model_type == 'gpt-j':
            model_name = f"EleutherAI/gpt-j-{model_size}"
        elif model_type == 'gpt-neo':
            model_name = f"EleutherAI/gpt-neo-{model_size}"
        elif model_type == 'Llama-3.2':
        # Llama 3.2 multilangl models
            model_name = f"meta-llama/Llama-3.2-{model_size}"
        elif model_type == "Qwen2.5":
            model_name = f"Qwen/Qwen2.5-{model_size}"
    
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"
    
        print(f"Modello: {model_type}")
        print(f"Nome completo: {model_name}")
        
        
        print(f"Caricamento modello: {model_name}")
        print(f"Algoritmo di decoding: {decoding_alg}")
        
        
        model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
        model.eval()
        
        print("Modello caricato con successo!")
        print(f"Device: {device}")
        
        
        
        
        for lang in languages:
            print("Language:", lang)
        
            dataset_path=f"{data_folder}/Dataset-{pii_type}-{lang}"
            if not os.path.exists(dataset_path):
                print("Data not ready at", dataset_path)
                continue
                
            
            data = load(pii_type, dataset_path)
            
        
        
            folder_name=f'{result_folder}/{pii_type}-{lang}'
            print(folder_name)
            
        
            torch.cuda.empty_cache()
            for x in contextToDo:
                print(f"\n{'='*80}")
                print(f"Setting: {x}")
                print('='*80)
        
                context_results_dir = f'{folder_name}/results-{x}'
                os.makedirs(context_results_dir, exist_ok=True)
                generated_file = f"{context_results_dir}/generated-{model_type}-{model_size}-{decoding_alg}.json"
                correct_file = f"{context_results_dir}/correct-{model_type}-{model_size}-{decoding_alg}.json"
                
                activation_results_dir = f"{folder_name}/activations-{x}/{model_type}-{model_size}-{decoding_alg}"
                os.makedirs(activation_results_dir, exist_ok=True)
            
                # Skip if already computed
                if not redo and os.path.exists(correct_file):
                    print(f"File {correct_file} already exists, skipped")
                    continue
            
                k = int(x.split('-')[-1])
            
                prompts, name_list = get_prompts_context(
                    dataset=data,
                    k=k
                )
            
                pii_found = defaultdict(str)
                generarated = defaultdict(str)
                processed_piis = set()
            
            
                for i in tqdm(
                    range(0, len(prompts), BATCH_SIZE),
                    desc="Generazione"
                ):
            
                    texts = prompts[i:i+BATCH_SIZE]
                    names_current = name_list[i:i+BATCH_SIZE]
            
            
                    encoding = tokenizer(
                        texts,
                        padding=True,
                        return_tensors="pt"
                    ).to(device)
            
            
                    with torch.no_grad():
            
                        if decoding_alg == "greedy":
            
                            outputs = model.generate(
                                **encoding,
                                pad_token_id=tokenizer.eos_token_id,
                                max_new_tokens=100,
                                do_sample=False,
                                temperature=None,
                                top_p=None,
                                return_dict_in_generate=True,
                                output_hidden_states=True
                            )
            
                        else:
                            print(f"{decoding_alg} not supported")
                            break
            
            
                    generated_ids = outputs.sequences
                    hidden_states = outputs.hidden_states
                    hidden_states = [
                        [hidden_states[i][j].to('cpu') for j in range(len(hidden_states[i]))] 
                        for i in range(len(hidden_states))
                    ]
                    torch.cuda.empty_cache()
                                     
                    #
                    # Reconstruct full sequence activations:
                    # [prompt tokens + generated tokens]
                    #
                    prompt_hidden = hidden_states[0]
                    n_layers = len(prompt_hidden)
            
                    all_hidden = []
                    for layer in range(n_layers):
                        layer_states = [
                            prompt_hidden[layer]
                        ]
            
                        # generated tokens
                        for step_hidden in hidden_states[1:]:
                            layer_states.append(
                                step_hidden[layer]
                            )
            
                        layer_states = torch.cat(
                            layer_states,
                            dim=1
                        ).to('cpu')
            
                        all_hidden.append(layer_states)
            
            
                    #
                    # Decode generated text for PII extraction
                    #
            
                    decoded = tokenizer.batch_decode(
                        generated_ids,
                        skip_special_tokens=True
                    )
            
            
                    batch_results = []
                    for j, s in enumerate(decoded):
                        # remove prompt
                        generated_text = s[len(texts[j]):]
                        batch_results.append(generated_text)                
            
                    #
                    # PII evaluation
                    #
            
                    for b, name, text in zip(
                        range(len(names_current)),
                        names_current,
                        batch_results
                    ):
            
                        processed_piis.add(name)
            
                        pii_in_example_found = pii_findall(text)
            
                        if pii_in_example_found:
                            #
                            # Check leak
                            #
                            if name in pii_in_example_found:
                                pii_found[name] = {
                                    "text": text,
                                    "piis": pii_in_example_found, 
                                    "index": b + i 
                                }
                                pii_index = pii_in_example_found.index(name)
                            else:
                                pii_index = 0
                            
        
                            generated_pii = pii_in_example_found[pii_index]
                            start_char = text.find(generated_pii)                
                            end_char = start_char + len(generated_pii)
        
                            
                            end = encoding['input_ids'].shape[-1] + len(tokenizer.encode(
                                text[:end_char]
                            ))
                            
                            
        
                            hidden_tensor = torch.stack(
                                [
                                    layer[b].cpu()
                                    for layer in all_hidden
                                ],
                                dim=0
                            )
        
                            
                            hidden_to_save = (
                                hidden_tensor[:, min(end,hidden_tensor.shape[1]-1)]
                                .to(torch.float16)
                                .cpu()
                            )
        
                            save_path = f"{activation_results_dir}/{names_current[b].replace('/', '-')}-{b + i}.pt"
                            torch.save(
                                {
                                    "generated_ids": generated_ids[b][:end].cpu(),
                                    "hidden_states": hidden_to_save,
                                    "generated_pii": generated_pii,
                                },
                                save_path,
                                pickle_protocol=5,
                            )
        
                            
        
                            #
                            # Save generation
                            #
                            generarated[name] = {
                                "text": text,
                                "piis": pii_in_example_found,
                                "index": b + i 
                            }
            
            
            
                    # free GPU memory
                    del outputs
                    del hidden_states
                    del generated_ids
                    torch.cuda.empty_cache()
            
            
            
                print("\n💾Saving final results...")
        
                save_json(generarated, generated_file) 
                save_json(pii_found, correct_file)
                
                print(f"   PII processed: {len(processed_piis)}")
                print(f"   PII found: {len(pii_found)}")
                print(
                    f"   Success rate: "
                    f"{len(pii_found)/len(processed_piis)*100:.2f}%")
    
        
        model = model.to('cpu')
        del model
        sleep(10)
        torch.torch.cuda.empty_cache()

In [ ]:
exit()